### Let's build the rdf graph with the provided csv

In [12]:
import pandas as pd
import urllib.parse
from rdflib import Graph, Literal, RDF, RDFS, URIRef, Namespace
from rdflib.namespace import XSD

df = pd.read_csv("Assignment2.csv")

g = Graph()

SCHEMA = Namespace("https://schema.org/")
EX = Namespace("http://example.org/webshop/")
PRODUCT = Namespace("http://example.org/webshop/product/")
BRAND = Namespace("http://example.org/webshop/brand/")
SUBCAT = Namespace("http://example.org/webshop/subcategory/")
CAT = Namespace("http://example.org/webshop/category/")

g.bind("schema", SCHEMA)
g.bind("ex", EX)
g.bind("product", PRODUCT)
g.bind("brand", BRAND)
g.bind("subcat", SUBCAT)
g.bind("cat", CAT)
g.bind("rdf", RDF)
g.bind("rdfs", RDFS)
g.bind("xsd", XSD)
s
category_map = {
    'InktCartridge': 'Supplies', 'TonerCartridge': 'Supplies',
    'Plotter': 'Printers', 'InkjetPrinter': 'Printers', 'Laserprinter': 'Printers',
    'Budgetlaptop': 'Laptops', 'Business Laptop': 'Laptops', 'Gaming Laptop': 'Laptops',
    'Gaming desktop': 'Desktops', 'workstation': 'Desktops'
}

for subc_name, cat_name in category_map.items():
    subc_uri = SUBCAT[urllib.parse.quote(subc_name.replace(" ", "_"))]
    cat_uri = CAT[urllib.parse.quote(cat_name)]
    g.add((subc_uri, RDFS.subClassOf, cat_uri))

for _, row in df.iterrows():
    raw_sku = str(row['schema_sku_value']).strip()
    clean_sku = urllib.parse.quote(raw_sku.replace(" ", "_"))

    item_uri = PRODUCT[clean_sku]
    brand_name = str(row['brand'])
    brand_uri = BRAND[urllib.parse.quote(brand_name)]
    subcat_name = str(row['subcategory']).replace(" ", "_")
    subcat_uri = SUBCAT[urllib.parse.quote(subcat_name)]

    g.add((item_uri, RDF.type, subcat_uri))
    g.add((item_uri, SCHEMA.sku, Literal(raw_sku, datatype=XSD.string)))
    g.add((item_uri, SCHEMA.name, Literal(str(row['item_name']), datatype=XSD.string)))
    g.add((item_uri, SCHEMA.url, URIRef(str(row['schema_url_value']))))
    g.add((item_uri, SCHEMA.price, Literal(float(row['schema_price_value']), datatype=XSD.decimal)))
    g.add((item_uri, SCHEMA.ratingValue, Literal(float(row['schema_rating_value']), datatype=XSD.float)))

    if pd.notnull(row['schema_color_value']):
        g.add((item_uri, SCHEMA.color, Literal(str(row['schema_color_value']), datatype=XSD.string)))

    g.add((item_uri, SCHEMA.brand, brand_uri))
    g.add((brand_uri, RDF.type, SCHEMA.Brand))
    g.add((brand_uri, SCHEMA.name, Literal(brand_name, datatype=XSD.string)))

g.serialize(destination="Group-6_ken3140_webshop.ttl", format="turtle")
print(f"RDF Graph created with {len(g)} triples and saved as 'Group-6_ken3140_webshop.ttl'.\n")


RDF Graph created with 418 triples and saved as 'Group-6_ken3140_webshop.ttl'.



### Adding the ontology to the populated rdfs graph for more freedom in queries
eg. select product instead of having to union everything.

In [14]:

from rdflib import Graph, Literal, RDF, RDFS, OWL, URIRef, Namespace

g = Graph()
g.parse("Group-6_ken3140_webshop.ttl", format="turtle")
g.parse("ontology.ttl", format="turtle")

SUBCAT = Namespace("http://example.org/webshop/subcategory/")
HP = Namespace("https://www.hp.com/ontology#")
SCHEMA = Namespace("https://schema.org/")

mappings = [
    (SUBCAT.Gaming_Laptop, RDFS.subClassOf, HP.GamingLaptop),
    (SUBCAT.Business_Laptop, RDFS.subClassOf, HP.BusinessLaptop),
    (SUBCAT.Budgetlaptop, RDFS.subClassOf, HP.BudgetLaptop),
    (SUBCAT.Gaming_desktop, RDFS.subClassOf, HP.GamingDesktop),
    (SUBCAT.workstation, RDFS.subClassOf, HP.Workstation),
    (SUBCAT.InkjetPrinter, RDFS.subClassOf, HP.InkjetPrinter),
    (SUBCAT.Laserprinter, RDFS.subClassOf, HP.LaserPrinter),
    (SUBCAT.Plotter, RDFS.subClassOf, HP.Plotter),
    (SUBCAT.InktCartridge, RDFS.subClassOf, HP.Ink),
    (SUBCAT.TonerCartridge, RDFS.subClassOf, HP.Ink),
    (SCHEMA.price, OWL.equivalentProperty, HP.price),
    (SCHEMA.brand, OWL.equivalentProperty, HP.hasBrand)
]

for triple in mappings:
    g.add(triple)
g.serialize(destination="Group-6_ken3140_webshop.ttl", format="turtle")

print(f"Merged Graph contains {len(g)} total triples.\n")

Merged Graph contains 510 total triples.



# Queries
a) For a given item (select a random item from your RDF Graph), provide all its
categories and subcategories, and its brand.


In [17]:
from rdflib import Graph, Literal
from rdflib.namespace import XSD

def run_query(query_str):
    res = g.query(query_str)
    for row in res:
        print([str(val) for val in row])
    print("\n")




sku = input("Please enter a product SKU: ").strip()

query = """
PREFIX schema: <https://schema.org/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?product ?productName ?brandName ?category WHERE {
    ?product schema:sku ?targetSku ;
             schema:name ?productName ;
             schema:brand ?brandResource .

    ?brandResource schema:name ?brandName .

    ?product a/rdfs:subClassOf* ?category .
}
"""# efficient because we first find the only object with a unique sku value then proceed to take information from it

results = g.query(
    query,
    initBindings={'targetSku': Literal(user_sku, datatype=XSD.string)}
)
for row in results:
    print([str(val) for val in row])
print("\n")

['http://example.org/webshop/product/CF372AM', 'HP 304A originele cyaan/magenta/gele LaserJet tonercartridge, 3-pack', 'LaserJet', 'http://example.org/webshop/subcategory/InktCartridge']
['http://example.org/webshop/product/CF372AM', 'HP 304A originele cyaan/magenta/gele LaserJet tonercartridge, 3-pack', 'LaserJet', 'http://example.org/webshop/category/Supplies']
['http://example.org/webshop/product/CF372AM', 'HP 304A originele cyaan/magenta/gele LaserJet tonercartridge, 3-pack', 'LaserJet', 'https://www.hp.com/ontology#Ink']
['http://example.org/webshop/product/CF372AM', 'HP 304A originele cyaan/magenta/gele LaserJet tonercartridge, 3-pack', 'LaserJet', 'https://schema.org/Product']




b) Provide items from different subcategories that have the same brand.


In [ ]:
#TODO use run query

c) Group products by brand and show the average price or rating for each brand.


In [ ]:
#TODO use run query

d) Sort products in one category according to average brand price or rating.


In [ ]:
#TODO use run query

e) Using an external service point (e.g. https://query.wikidata.org/), provide a
description of 5 facts about the top ranked brand from part D, e.g. location of
headquarters. You may return images as one of your facts.


In [ ]:
#TODO use run query

f) Recommend an item which is similar to the item using your linked RDF graph
(i.e., shared properties and categories).


In [ ]:
#TODO use run query

g) Write your own question about the webshop in plain English, then translate it to
the corresponding SPARQL query, and run it on the graph. Provide a rationale for
why this query would be valuable in a webshop setting, such as for semantic
search or other applications.

In [ ]:
#TODO use run query